# Explore County Health Rankings analytic data columns

The files in `data/analytic_data<year>.csv` are County Health Rankings & Roadmaps "analytic data" exports. Column counts vary a lot year to year (198 in 2010 up to 796 in 2025) as CHR adds new measures and demographic breakouts. This notebook loads the headers only (fast, no need to parse full files) and helps us understand what's available.

In [ ]:
import csv
import glob
import os
import re
from collections import defaultdict

DATA_DIR = "data"
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "analytic_data*.csv")))
csv_files

In [ ]:
def read_header(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return next(csv.reader(f))

headers_by_year = {}
for path in csv_files:
    year = re.search(r"(\d{4})", os.path.basename(path)).group(1)
    headers_by_year[year] = read_header(path)

{year: len(cols) for year, cols in headers_by_year.items()}

## Columns for a single year

Pick a year and list every column name.

In [ ]:
YEAR = "2025"
cols = headers_by_year[YEAR]
print(f"{len(cols)} columns in {YEAR}\n")
for c in cols:
    print(c)

## Search for columns by keyword

Case-insensitive substring search across a chosen year's columns — handy for finding e.g. all "Veteran" or "raw value" fields.

In [ ]:
def search_columns(year, keyword):
    keyword_lower = keyword.lower()
    return [c for c in headers_by_year[year] if keyword_lower in c.lower()]

search_columns("2025", "veteran")

## Group columns into base measures

Each CHR measure typically expands into several columns (raw value, numerator, denominator, CI low/high, flags, and per-race breakouts). Strip the common suffixes to recover the underlying measure names.

In [ ]:
SUFFIX_PATTERN = re.compile(
    r"\s*(raw value|numerator|denominator|CI low.*|CI high.*|flag.*|"
    r"\((AIAN|Asian.*|Black|Hispanic|White|NHOPI|Two or more races)\).*)$",
    re.IGNORECASE,
)

def base_measure(col):
    return SUFFIX_PATTERN.sub("", col).strip()

def measures_for_year(year):
    grouped = defaultdict(list)
    for c in headers_by_year[year]:
        grouped[base_measure(c)].append(c)
    return grouped

grouped_2025 = measures_for_year("2025")
print(f"{len(grouped_2025)} distinct base measures in 2025\n")
for measure, variants in sorted(grouped_2025.items()):
    print(f"{measure}  ({len(variants)} columns)")

## Compare columns across years

Which columns are new, dropped, or shared between two given years?

In [ ]:
def compare_years(year_a, year_b):
    set_a, set_b = set(headers_by_year[year_a]), set(headers_by_year[year_b])
    return {
        "shared": sorted(set_a & set_b),
        "only_in_a": sorted(set_a - set_b),
        "only_in_b": sorted(set_b - set_a),
    }

diff = compare_years("2024", "2025")
print(f"Shared: {len(diff['shared'])}")
print(f"Only in 2024: {len(diff['only_in_a'])}")
print(f"Only in 2025: {len(diff['only_in_b'])}")

In [ ]:
print("New in 2025:")
for c in diff["only_in_b"]:
    print(" ", c)

print("\nDropped after 2024:")
for c in diff["only_in_a"]:
    print(" ", c)

## Columns present in every year

The stable core you can safely rely on for a multi-year time series.

In [ ]:
first_year = next(iter(headers_by_year))
common_cols = set(headers_by_year[first_year])
for year in headers_by_year:
    common_cols &= set(headers_by_year[year])

print(f"{len(common_cols)} columns present in all {len(headers_by_year)} years\n")
for c in sorted(common_cols):
    print(c)

## Raw value columns only

Each measure has a `... raw value` column plus a bunch of `numerator`/`denominator`/`CI low`/`CI high`/`flag` and per-race variant columns alongside it. If you just want the headline value per measure, filter down to columns containing `"raw value"` — checked against 2025's header, this substring cleanly grabs the 90 main measure columns with zero false positives from the race-breakout columns (those use ` (Black)`, ` (AIAN)`, etc. without the words "raw value").

In [ ]:
def raw_value_columns(year):
    return [c for c in headers_by_year[year] if "raw value" in c.lower()]

YEAR = "2025"
raw_cols = raw_value_columns(YEAR)
print(f"{len(raw_cols)} raw value columns in {YEAR}\n")
for c in raw_cols:
    print(c.replace(" raw value", ""))

## Load a year's data with only ID + raw value columns

Loads the full CSV for one year but keeps only the geography/identifier columns plus every `raw value` column — skips numerators, denominators, CIs, flags, and race breakouts entirely.

In [ ]:
import pandas as pd

ID_COLS = [
    "State FIPS Code",
    "County FIPS Code",
    "5-digit FIPS Code",
    "State Abbreviation",
    "Name",
    "Release Year",
]

def load_raw_values(year):
    path = os.path.join(DATA_DIR, f"analytic_data{year}.csv")
    cols = ID_COLS + raw_value_columns(year)
    # Row 1 (0-indexed) is a second header of machine variable codes
    # (e.g. "v058_rawvalue"), not data -- skip it.
    # low_memory=False also avoids a pandas C-parser bug that trips over the
    # mixed-dtype warning path when usecols narrows a very wide file like this.
    df = pd.read_csv(path, usecols=cols, encoding="utf-8-sig", skiprows=[1], low_memory=False)
    return df[cols]

df_2025 = load_raw_values("2025")
print(df_2025.shape)
df_2025.head()